In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif, RFE

from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_curve, roc_auc_score, precision_recall_curve, auc)


In [4]:
df = pd.read_csv(r'../merge/raw/csv_full.csv', sep=';')

null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)

pd.DataFrame({
    'null_count': null_counts,
    'null_%': null_pct
}).sort_values('null_%', ascending=False)

,null_count,null_%
taux_plein_teom,54845,56.45
taux_global_th,49879,51.34
taux_global_tfb,49879,51.34
taux_global_tfnb,49879,51.34
prix_m2,5033,5.18
zone_emploi,142,0.15
population,7,0.01
code_postal,7,0.01
dep_code,7,0.01
nom_commune,7,0.01


In [5]:
df_model = df.drop(columns=['nom_commune', 'dep_nom', 'reg_nom'])

df_model.head()

,annee,code_commune,prix_m2,dep_code,reg_code,code_postal,population,densite,superficie_km2,zone_emploi,taux_global_tfb,taux_global_tfnb,taux_plein_teom,taux_global_th,y
0,2022,1001,2930.586507,1.0,84.0,1400.0,779.0,48.7,16.0,8405.0,38.82,118.47,9.95,28.63,stable
1,2022,1002,2575.603876,1.0,84.0,1640.0,256.0,27.1,9.0,8405.0,38.82,118.47,9.95,28.63,stable
2,2022,1004,3452.219201,1.0,84.0,1500.0,14134.0,570.5,24.0,8405.0,38.82,118.47,9.95,28.63,stable
3,2022,1005,11225.915649,1.0,84.0,1330.0,1751.0,106.1,16.0,8434.0,38.82,118.47,9.95,28.63,stable
4,2022,1006,1421.052632,1.0,84.0,1300.0,112.0,18.9,6.0,8404.0,38.82,118.47,9.95,28.63,stable


In [6]:
df_model.columns.to_list()

['annee',
 'code_commune',
 'prix_m2',
 'dep_code',
 'reg_code',
 'code_postal',
 'population',
 'densite',
 'superficie_km2',
 'zone_emploi',
 'taux_global_tfb',
 'taux_global_tfnb',
 'taux_plein_teom',
 'taux_global_th',
 'y']

In [13]:
features = ['annee', 'code_commune', 'dep_code', 'reg_code', 'code_postal', 'population', 'densite', 'superficie_km2',
 'zone_emploi', 'taux_global_tfb', 'taux_global_tfnb', 'taux_plein_teom', 'taux_global_th']

X = df_model[features]
y = df_model['y']

train = df_model[df_model['annee'] < 2024]
test  = df_model[df_model['annee'] == 2024]

X_train = train[features]
y_train = train['y']
X_test  = test[features]
y_test  = test['y']

print(f"Train : {len(X_train)} lignes")
print(f"Test  : {len(X_test)} lignes")
print(f"\nDistribution train :\n{y_train.value_counts()}")
print(f"\nDistribution test :\n{y_test.value_counts()}")

Train : 64892 lignes
Test  : 32266 lignes

Distribution train :
y
stable    35401
baisse    15307
hausse    14184
Name: count, dtype: int64

Distribution test :
y
baisse    15487
hausse    13517
stable     3262
Name: count, dtype: int64


In [20]:
pipeline_simple = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(f_classif, k=5)),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100))
])

pipeline_simple.fit(X_train, y_train)

# Prédictions
y_pred = pipeline_simple.predict(X_test)

# Évaluation
accuracy = accuracy_score(y_test, y_pred)
print(f"✓ Pipeline entraîné !")
print(f"Accuracy : {accuracy:.2%}")

✓ Pipeline entraîné !
Accuracy : 38.82%


In [21]:
print(classification_report(y_test, y_pred))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred, labels=['hausse', 'baisse', 'stable'])


              precision    recall  f1-score   support

      baisse       0.41      0.42      0.41     15487
      hausse       0.33      0.33      0.33     13517
      stable       0.60      0.48      0.53      3262

    accuracy                           0.39     32266
   macro avg       0.44      0.41      0.42     32266
weighted avg       0.39      0.39      0.39     32266

